In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Chargement des données
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

# 2. Préparation des données (avec la correction Pandas 2/3)
categorical = list(df.select_dtypes(include=['object', 'string']).columns)
numerical = list(df.select_dtypes(include=['number']).columns)

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

# --- Q1 ---
mode_industry = df['industry'].mode()[0]
print(f"Q1 - Mode pour 'industry' : {mode_industry}")

# --- Q2 ---
num_features = [col for col in numerical if col != 'converted']
corr_matrix = df[num_features].corr()

pairs_to_check = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

print("\nQ2 - Corrélation entre les paires spécifiées :")
for f1, f2 in pairs_to_check:
    if f1 in corr_matrix.columns and f2 in corr_matrix.columns:
        print(f"  {f1} & {f2}: {corr_matrix.loc[f1, f2]:.4f}")

# --- Étape Split ---
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

# --- Q3 ---
categorical_features = [c for c in categorical if c != 'converted']

def calculate_mi(series):
    return mutual_info_score(series, y_train)

mi_scores = df_train[categorical_features].apply(calculate_mi)
mi_scores = round(mi_scores, 2)

print("\nQ3 - Mutual Information scores :")
print(mi_scores.sort_values(ascending=False))

# --- Q4 ---
all_features = categorical_features + num_features

dv = DictVectorizer(sparse=False)
train_dict = df_train[all_features].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[all_features].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
original_accuracy = accuracy_score(y_val, y_pred)

print(f"\nQ4 - Précision du modèle : {round(original_accuracy, 2)}")

# --- Q5 ---
feature_candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

print("\nQ5 - Différences de précision après élimination :")
for feat in feature_candidates:
    subset_features = [f for f in all_features if f != feat]
    
    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(df_train[subset_features].to_dict(orient='records'))
    X_val_sub = dv_sub.transform(df_val[subset_features].to_dict(orient='records'))
    
    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)
    
    y_pred_sub = model_sub.predict(X_val_sub)
    acc_sub = accuracy_score(y_val, y_pred_sub)
    diff = original_accuracy - acc_sub
    
    print(f"  Sans '{feat}': Différence = {diff:.5f} (Acc = {acc_sub:.5f})")

# --- Q6 ---
C_values = [0.000001, 0.00001, 0.0001, 0.001]

print("\nQ6 - Évaluation des valeurs de C :")
for C in C_values:
    model_c = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model_c.fit(X_train, y_train)
    
    y_pred_c = model_c.predict(X_val)
    acc_c = accuracy_score(y_val, y_pred_c)
    
    print(f"  C = {C:<8} -> Précision = {round(acc_c, 3)}")